In [0]:
from pyspark.sql.functions import lag, col, avg
from pyspark.sql.window import Window

silver_df = spark.read.table("workspace.stock_data.stock_data_silver")

window_spec = Window.partitionBy("symbol").orderBy("date")

gold_df = (
    silver_df
    .withColumn("prev_close", lag("close").over(window_spec))
    .withColumn("daily_return", (col("close") - col("prev_close")) / col("prev_close"))
    .withColumn("sma_20", avg("close").over(window_spec.rowsBetween(-19, 0)))
    .drop("prev_close")
)

gold_df.write.format("delta").mode("overwrite").saveAsTable("workspace.stock_data.stock_data_gold")

In [0]:
%sql
SELECT *
FROM workspace.stock_data.stock_data_gold